# AnyJev на CPU: локальный typed decision evaluator

Отдельный эксперимент, не связанный с deterministic retrieval aggregation и DeepEval.
Он запускает training-free `AnyJev Decider` через `llama.cpp` полностью на CPU и возвращает
распределение вероятностей для `choice`, `noul` и `score`.

Важные ограничения:

- `anyjev` и `llama-cpp-python` пока **не входят** в зависимости `mvp_v2`;
- ноутбук не устанавливает пакеты и не изменяет `.venv`;
- перед запуском зависимости нужно добавить в `pyproject.toml` и синхронизировать внешнее окружение;
- путь к уже скачанному GGUF задаётся в `GGUF_PATH`;
- `L0` делает перестановки вариантов и медленнее `raw`, особенно на CPU;
- полученные вероятности нужно валидировать на размеченных данных; это не замена `Hit@k`/`MRR`.


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import os
import platform
import sys
import time
from pathlib import Path

REQUIRED_PACKAGES = ("anyjev", "llama-cpp-python")
missing = []
for package in REQUIRED_PACKAGES:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        missing.append(package)
    else:
        print(f"{package}: {version}")

print("Python:", sys.version.split()[0], sys.executable)
print("Platform:", platform.platform())
print("Logical CPU cores:", os.cpu_count())
if missing:
    raise RuntimeError(
        "Не установлены зависимости: " + ", ".join(missing) + ". "
        "Добавьте anyjev[llamacpp]==0.3.0 в pyproject.toml, "
        "выполните uv sync для внешнего mvp_v2-окружения, "
        "затем Kernel → Restart Kernel."
    )


## 1. Воспроизводимая конфигурация


In [ ]:
# Полный путь к уже скачанному GGUF-файлу.
# Пример: Path("/Users/zero/models/qwen2.5-0.5b-instruct-q4_k_m.gguf")
GGUF_PATH = Path("")

N_CONTEXT = 2_048
N_BATCH = 256
N_THREADS = max(1, (os.cpu_count() or 2) // 2)
LEVEL = "L0"  # "raw" быстрее, но сильнее зависит от порядка вариантов.

if LEVEL not in {"raw", "L0"}:
    raise ValueError("LEVEL должен быть raw или L0")
print("CPU threads:", N_THREADS, "| level:", LEVEL)


## 2. Проверка локального GGUF


In [ ]:
model_path = GGUF_PATH.expanduser()
if not str(GGUF_PATH):
    raise ValueError("Укажите полный путь к GGUF в переменной GGUF_PATH")
if not model_path.is_absolute():
    raise ValueError(f"GGUF_PATH должен быть абсолютным: {model_path}")
if not model_path.is_file():
    raise FileNotFoundError(f"GGUF не найден: {model_path}")
if model_path.suffix.casefold() != ".gguf":
    raise ValueError(f"Ожидался файл с расширением .gguf: {model_path}")

model_bytes = model_path.stat().st_size
print("Model:", model_path)
print("Size:", f"{model_bytes / 1024**2:.1f} MiB")


## 3. CPU backend и `Decider`


In [ ]:
from anyjev import Decider, Question
from anyjev.backends.llamacpp import LlamaCppBackend

started_at = time.perf_counter()
backend = LlamaCppBackend(
    str(model_path),
    n_ctx=N_CONTEXT,
    n_batch=N_BATCH,
    n_gpu_layers=0,
    n_threads=N_THREADS,
    verbose=False,
)
decider = Decider(backend, level=LEVEL)
print("Loaded in", round(time.perf_counter() - started_at, 2), "s")


## 4. Choice, Noul и Score


In [ ]:
state = "Клиент сообщает, что с карты дважды списали деньги за один заказ."
questions = [
    Question.choice(
        "Какая команда должна обработать обращение?",
        ["billing", "technical", "sales", "other"],
        name="route",
    ),
    Question.noul(
        "Относится ли обращение к проблеме оплаты?",
        name="is_payment_issue",
    ),
    Question.score(
        "Насколько срочно нужно обработать обращение?",
        levels=["низкая", "средняя", "высокая"],
        name="urgency",
    ),
]

started_at = time.perf_counter()
decisions = decider.decide(state, questions, require=LEVEL)
elapsed_s = time.perf_counter() - started_at
payload = decisions.to_dict()
print(json.dumps(payload, ensure_ascii=False, indent=2))
print("Decision time:", round(elapsed_s, 2), "s")

for decision in decisions:
    assert abs(sum(decision.distribution.values()) - 1.0) < 1e-6
    assert decision.level == LEVEL


## 5. Пакетная проверка


In [ ]:
# Минимальный шаблон для пакетной проверки одного typed question.
route_question = questions[0]
states = [
    "Платёж списан дважды.",
    "Не открывается приложение после обновления.",
    "Хочу узнать стоимость корпоративного тарифа.",
]

started_at = time.perf_counter()
batch = decider.decide_batch(states, route_question, require=LEVEL)
rows = [
    {
        "state": state,
        "answer": decision.answer,
        "confidence": decision.confidence,
        "distribution": decision.distribution,
        "level": decision.level,
    }
    for state, decision in zip(states, batch, strict=True)
]
print(json.dumps(rows, ensure_ascii=False, indent=2))
print("Batch time:", round(time.perf_counter() - started_at, 2), "s")


## Интерпретация и следующий эксперимент

- `raw` — один readout, быстрый, но чувствителен к позиции и метке варианта.
- `L0` — marginalization по перестановкам и label-free prior correction.
- Один демонстрационный вызов не доказывает calibration.
- Для сравнения с DeepEval нужно заранее зафиксировать typed rubric и размеченный датасет,
  затем считать accuracy, Brier score, log loss, ECE и устойчивость к перестановкам.
- Для воспроизводимости отдельно зафиксируйте имя и SHA-256 своего GGUF-файла.
- В ноутбуке зафиксированы AnyJev `0.3.0`, CPU-only `n_gpu_layers=0`,
  контекст, batch и число потоков.


In [ ]:
# Освободить mmap/KV-cache перед повторной загрузкой модели.
backend.close()
